# Notebook overview: R8_2-CO_Randomremoval_interaction.ipynb

## What this notebook does
Constructs evacuation-controlled random survival counterfactuals for the Colorado co-location network. It aligns observed dyads with evacuation status and home CBG information, samples surviving users by CBG, and prepares random-removal graphs/metrics for comparison with the observed post-disaster network.

## Inputs used
- Dyadic interaction table: ../Data/stop_df_perday_CO/for_ABM/individual_interac_allusers_{revision}.csv
- User evacuation file: ../Data/stop_df_perday_CO/graphs_POI_weighted/user_evacuations/user_evacuations_{revision}.csv
- Home CBG files under ../Data/stop_df_perday_CO/home/
- Observed pre/post graph pickles under ../Data/stop_df_perday_CO/graphs_POI_weighted/
- Census/CBG demographic files used for CBG-level matching

## Outputs created
- Aligned dyad table: ../Data/stop_df_perday_CO/for_ABM/individual_interac_allusers_{revision}_aligned.csv
- Random survivor lists by run
- Random-survival counterfactual graph pickles and network metric tables under graphs_POI_weighted/boots_* directories

**Data access warning.** The Cuebiq/Spectus mobility stop data used by this project are proprietary/restricted and are not included in this repository. Do not commit raw mobility files, user IDs, stop tables, home-location files, graph outputs containing sensitive identifiers, or large derived files unless your data-use agreement explicitly permits release. Public users must obtain data access independently and place files in the documented paths.

In [ ]:
pip install statsmodels 

In [ ]:
pip install seaborn

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from sklearn.neighbors import BallTree
import numpy as np
from scipy.sparse import lil_matrix
import json
from collections import defaultdict
import networkx as nx
import pickle
import os
import matplotlib.pyplot as plt
from shapely.geometry import Point
from shapely import wkt
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import r2_score
import seaborn as sns

In [ ]:
print("Current working directory:", os.getcwd())
base = os.path.join("..", "Data", "stop_df_perday_CO")
pois_dir = os.path.join(base, "POIs")
geo_dir = os.path.join(base, "geography_CO")
stops_dir = os.path.join(base, "daily_agg_to_weekly_Stops")
graph_poi_dir = os.path.join(base, "graphs", "poi-user")
os.makedirs(graph_poi_dir, exist_ok=True)
week_dir = os.path.join(graph_poi_dir, "user_connections" )
os.makedirs(week_dir, exist_ok=True)
home_dir = os.path.join(base, "home/pre_disaster")

revision = "R11"

boot_dir = os.path.join(graph_poi_dir, "boots_user_removal", revision)
cf_out_dir = os.path.join(graph_poi_dir, "counterfactual_post", revision)

os.makedirs(cf_out_dir, exist_ok=True)

survivor_dir = os.path.join(graph_poi_dir, "boots_user_survivors", revision)
os.makedirs(survivor_dir, exist_ok=True)

# Load required Data

In [ ]:
dyad_path =  os.path.join(base, "for_ABM", f"individual_interac_allusers_{revision}.csv")
dyad_poi_wide = pd.read_csv(dyad_path)

In [ ]:
dyad_poi_wide.head()

In [ ]:
week_dir = os.path.join(graph_poi_dir, "user_evacuations" )
os.makedirs(week_dir, exist_ok=True)
output_path = os.path.join(week_dir, f"user_evacuations_{revision}.csv")
user_evac_df = pd.read_csv(output_path)
user_evac_df

In [ ]:
# --- ensure user IDs are strings ---
dyad_poi_wide["user_i"] = dyad_poi_wide["user_i"].astype(str)
dyad_poi_wide["user_j"] = dyad_poi_wide["user_j"].astype(str)

user_evac_df["user"] = user_evac_df["user"].astype(str)

# --- users present in dyads ---
users_in_dyads = set(pd.concat([dyad_poi_wide["user_i"], dyad_poi_wide["user_j"]]).unique())

# --- users present in evac table ---
users_in_evac = set(user_evac_df["user"].unique())

# --- overlap ---
users_overlap = users_in_dyads.intersection(users_in_evac)

print("Unique users in dyads:", len(users_in_dyads))
print("Unique users in evac table:", len(users_in_evac))
print("Overlap users:", len(users_overlap))
print("Evac users missing from dyads:", len(users_in_evac - users_in_dyads))
print("Dyad users missing from evac table:", len(users_in_dyads - users_in_evac))

# --- within overlap: evac vs remain ---
overlap_evac = user_evac_df.loc[user_evac_df["user"].isin(users_overlap), ["user", "evacuation"]]

n_evac = int((overlap_evac["evacuation"] == 1).sum())
n_remain = int((overlap_evac["evacuation"] == 0).sum())

print("\nAmong overlap users:")
print("  Evacuated:", n_evac)
print("  Remained:", n_remain)
print("  Total:", n_evac + n_remain)

In [ ]:
# Drop Dyads Containing Users Not in Evac Table
# valid users universe
valid_users = set(user_evac_df["user"])

# keep dyads where BOTH users are in evac universe
dyad_aligned = dyad_poi_wide.loc[
    dyad_poi_wide["user_i"].isin(valid_users) &
    dyad_poi_wide["user_j"].isin(valid_users)
].copy()

print("Dyads after alignment:", len(dyad_aligned))

# sanity check
users_after = set(pd.concat([dyad_aligned["user_i"],
                             dyad_aligned["user_j"]]))

print("Unique users after alignment:", len(users_after))
print("Should equal evac universe:", len(valid_users))

In [ ]:
# Build lookup dictionary
evac_lookup = dict(zip(user_evac_df["user"], user_evac_df["evacuation"]))

# Map evacuation status
dyad_aligned["evac_status_i"] = dyad_aligned["user_i"].map(evac_lookup)
dyad_aligned["evac_status_j"] = dyad_aligned["user_j"].map(evac_lookup)

# Sanity check: no missing
print("Missing evac_status_i:", dyad_aligned["evac_status_i"].isna().sum())
print("Missing evac_status_j:", dyad_aligned["evac_status_j"].isna().sum())

In [ ]:
dyad_aligned.head()

In [ ]:
interactions = os.path.join(base, "for_ABM", f"individual_interac_allusers_{revision}_aligned.csv")
dyad_aligned.to_csv(interactions, index = False)


In [ ]:
start_date = 20211001
end_date = 20211131
freq_home_path = os.path.join(home_dir, f"freq_home_{start_date}_{end_date}")
home_df = pd.read_csv(freq_home_path)
home_df = home_df.rename(columns={"cuebiq_id": "user", "fips_code": "pre_disaster_fips"})
home_df.columns

In [ ]:
# home_df["pre_disaster_home"] = home_df["pre_disaster_home"].apply(
#     lambda x: str(int(x)) if pd.notnull(x) else pd.NA
# )
home_df['pre_disaster_fips']

# Prepping data

In [ ]:
# user_evac_df: columns ["user","evacuation"]
user_evac_df["user"] = user_evac_df["user"].astype(str)

# home_df: columns ["user","pre_disaster_home"]
home_df = home_df.copy()
home_df["user"] = home_df["user"].astype(str)

ue = user_evac_df.merge(home_df[["user", "pre_disaster_fips"]], on="user", how="left")

ue

In [ ]:
# Keep only users with known home CBG (recommended for clean constraints)
ue = ue.dropna(subset=["pre_disaster_fips"]).copy()
ue["pre_disaster_fips"] = ue["pre_disaster_fips"].astype(str)

ue

In [ ]:
# Pre pool per CBG (all pre users with home)
pool_by_cbg = ue.groupby("pre_disaster_fips")["user"].apply(np.array)

# Observed evac counts per CBG (L_g)
L_by_cbg = ue.loc[ue["evacuation"] == 1].groupby("pre_disaster_fips").size().astype(int)

# Total pre users per CBG (N_g)
N_by_cbg = ue.groupby("pre_disaster_fips").size().astype(int)

# Sanity: L_g <= N_g
bad = (L_by_cbg > N_by_cbg.reindex(L_by_cbg.index)).sum()
print("CBGs with L_g > N_g:", bad)

# Sratified Random Removal

How is the impact of users who actually left, different from the ones that are randomly removed? 

Create a sample $S_g$ = $E_g$ in count, so every user $i$ in a CBG $g$ has equal probability = $ E_g / T_g $ in $g$; $E_g$ = Observed evacuation, $T_g$ - Total users

In [ ]:
def draw_random_evacuations(pool_by_cbg, L_by_cbg, rng: np.random.Generator):
    """
    Returns:
      removed_set: set of users removed in this draw
      removed_by_cbg: dict cbg -> list(users removed)
    """
    removed_by_cbg = {}
    removed_set = set()

    for cbg, users in pool_by_cbg.items():
        Lg = int(L_by_cbg.get(cbg, 0))
        if Lg <= 0:
            removed_by_cbg[cbg] = []
            continue

        users = np.array(users, dtype=object)
        Lg = min(Lg, len(users))  # safety
        pick = rng.choice(users, size=Lg, replace=False)
        removed_by_cbg[cbg] = pick.tolist()
        removed_set.update(pick.tolist())

    return removed_set, removed_by_cbg

In [ ]:
def run_user_randomization(ue, pool_by_cbg, L_by_cbg, n_runs=500, seed=0, ci=(2.5, 97.5)):
    """
    Produces:
      draws_long: (run, user, removed 0/1)
      user_ci: per-user removal prob + percentile CI
      cbg_checks: per-run checks on removal counts
    """
    rng = np.random.default_rng(seed)

    users_all = ue["user"].astype(str).unique()
    user_to_idx = {u:i for i,u in enumerate(users_all)}
    M = len(users_all)

    # store draws as boolean matrix [n_runs, M]
    removed_mat = np.zeros((n_runs, M), dtype=np.uint8)

    cbg_checks = []

    for r in range(n_runs):
        removed_set, removed_by_cbg = draw_random_evacuations(pool_by_cbg, L_by_cbg, rng)

        # fill matrix
        idx = [user_to_idx[u] for u in removed_set if u in user_to_idx]
        removed_mat[r, idx] = 1

        # optional validation: counts match L_g
        # (this is usually exact given our draw function)
        chk = {
            "run": r,
            "removed_total": int(removed_mat[r].sum()),
            "target_total": int(L_by_cbg.sum())
        }
        cbg_checks.append(chk)

    # per-user stats
    p_hat = removed_mat.mean(axis=0)
    lo = np.percentile(removed_mat, ci[0], axis=0)
    hi = np.percentile(removed_mat, ci[1], axis=0)

    user_ci = pd.DataFrame({
        "user": users_all,
        "p_removed_mean": p_hat,
        f"p_removed_p{ci[0]}": lo,
        f"p_removed_p{ci[1]}": hi
    })

    cbg_checks = pd.DataFrame(cbg_checks)
    return removed_mat, user_ci, cbg_checks

In [ ]:
removed_mat, user_ci, cbg_checks = run_user_randomization(
    ue=ue,
    pool_by_cbg=pool_by_cbg,
    L_by_cbg=L_by_cbg,
    n_runs=500,
    seed=42
)

print(cbg_checks.head())
print(user_ci.sort_values("p_removed_mean", ascending=False).head(10))

## Export all runs - removed

In [ ]:
users_all = ue["user"].astype(str).unique()

def keep_set_from_removed_mat_row(removed_row, users_all):
    removed = set(users_all[np.where(removed_row == 1)[0]])
    return set(users_all) - removed

# example: keep set for run 0
keep1 = keep_set_from_removed_mat_row(removed_mat[1], users_all)
len(keep1)

In [ ]:
boot_dir = os.path.join(graph_poi_dir, "boots_user_removal", revision)
os.makedirs(boot_dir, exist_ok=True)

for r in range(removed_mat.shape[0]):
    removed_users = users_all[np.where(removed_mat[r] == 1)[0]].tolist()
    with open(os.path.join(boot_dir, f"removed_users_run{r:04d}.json"), "w") as f:
        json.dump(removed_users, f)

In [ ]:
# build lookup: user -> cbg - verify exact CBG constraints per run
user_to_cbg = dict(zip(ue["user"], ue["pre_disaster_fips"]))

def check_counts_one_run(removed_users):
    s = pd.Series([user_to_cbg.get(u, None) for u in removed_users]).dropna()
    c = s.value_counts()
    # compare to L_by_cbg
    diff = (c.reindex(L_by_cbg.index, fill_value=0) - L_by_cbg).abs().sum()
    return diff

# should be 0 always
removed0 = users_all[np.where(removed_mat[0] == 1)[0]].tolist()
print("diff run0:", check_counts_one_run(removed0))

# Export all runs - Survivors

In [ ]:
# Actual survivors from data
actual_survivors = set(
    ue.loc[ue["evacuation"] == 0, "user"].astype(str)
)

actual_survivor_count = len(actual_survivors)
print("Actual survivors:", actual_survivor_count)

In [ ]:
users_all = ue["user"].astype(str).unique()
users_all = np.array(users_all, dtype=object)

survivor_counts = []

for r in range(removed_mat.shape[0]):

    removed_idx = np.where(removed_mat[r] == 1)[0]
    removed_set = set(users_all[removed_idx])

    survivors_set = set(users_all) - removed_set

    survivor_counts.append(len(survivors_set))

    # Save survivors
    with open(os.path.join(boot_dir, f"survivors_run{r:04d}.json"), "w") as f:
        json.dump(list(survivors_set), f)

print("Saved 500 survivor sets.")

In [ ]:
print("Actual survivor count:", actual_survivor_count)
print("Mean simulated survivor count:", np.mean(survivor_counts))
print("Min simulated survivor count:", np.min(survivor_counts))
print("Max simulated survivor count:", np.max(survivor_counts))

# Testing if random and post have same # of nodes per cbg

In [ ]:
ue2 = ue.copy()
ue2["user"] = ue2["user"].astype(str)
ue2 = ue2.dropna(subset=["pre_disaster_fips"]).copy()

ue2["pre_disaster_fips"] = ue2["pre_disaster_fips"].astype(str)

pre_by_cbg = ue2.groupby("pre_disaster_fips")["user"].nunique()
post_by_cbg = ue2.loc[ue2["evacuation"] == 0].groupby("pre_disaster_fips")["user"].nunique()

cbg_index = pre_by_cbg.index.union(post_by_cbg.index)

In [ ]:
cbg_index

In [ ]:
rand_counts = []
for r in range(500):
    path = os.path.join(boot_dir, f"survivors_run{r:04d}.json")
    with open(path, "r") as f:
        surv = set(map(str, json.load(f)))
    c = ue2.loc[ue2["user"].isin(surv)].groupby("pre_disaster_fips")["user"].nunique()
    rand_counts.append(c.reindex(cbg_index, fill_value=0).astype(float))

rand_mean_by_cbg = pd.concat(rand_counts, axis=1).mean(axis=1)
rand_mean_by_cbg

In [ ]:
def clean_cbg_geoid(x):
    s = str(x).strip()
    s = s.replace(".0", "")
    s = s.split(".")[0]
    return s.zfill(12)

cbg_summary_boulder = cbg_summary_df.copy()
cbg_summary_boulder["cbg_clean"] = cbg_summary_boulder["cbg"].map(clean_cbg_geoid)

cbg_summary_boulder = cbg_summary_boulder[
    cbg_summary_boulder["cbg_clean"].str.startswith("08013")
].copy()

cbg_summary_boulder["cbg"] = cbg_summary_boulder["cbg_clean"]
cbg_summary_boulder = cbg_summary_boulder.drop(columns=["cbg_clean"]).reset_index(drop=True)

cbg_summary_boulder

In [ ]:
plot_df = cbg_summary_boulder[["post_users_n", "random_users_mean_n"]].copy()
plot_df["post_users_n"] = pd.to_numeric(plot_df["post_users_n"], errors="coerce")
plot_df["random_users_mean_n"] = pd.to_numeric(plot_df["random_users_mean_n"], errors="coerce")
plot_df = plot_df.dropna()

plt.figure(figsize=(6.5, 4))
plt.hist(plot_df["random_users_mean_n"].values, bins=40, alpha=0.8, label="Random (mean across runs)")
plt.hist(plot_df["post_users_n"].values, bins=40, alpha=0.55, label="Actual Post")
plt.xlabel("Users per CBG")
plt.ylabel("Count of CBGs")
plt.legend()
plt.tight_layout()
plt.show()

# Random Counterfactual - user - user ties

Random post = induced subgraph of pre: 
For a given random removal draw b with survivor set $S_b$ : a dyad–POI record (u,v,poi) survives if : $ u \in S_b $ and $v \in S_b$

## Load actual pre

In [ ]:
dyad_path =  os.path.join(base, "for_ABM", f"individual_interac_allusers_{revision}_aligned.csv")
dy = pd.read_csv(dyad_path)
dy

In [ ]:
# ensure strings
dy["user_i"] = dy["user_i"].astype(str)
dy["user_j"] = dy["user_j"].astype(str)
dy["poi"]    = dy["poi"].astype(str)

# pre-only weight per dyad-poi
dy["pre_sum"] = dy[["Oct2021", "Nov2021"]].sum(axis=1)

# keep only dyads that actually exist in pre (optional but speeds up)
dy_pre = dy.loc[dy["pre_sum"] > 0, ["user_i","user_j","poi","pre_sum", "evac_status_i", "evac_status_j"]].copy()
print("dy_pre rows:", len(dy_pre))
dy_pre.head()

In [ ]:
poi_pre = ( # build poi level stats for pre
    dy_pre
    .groupby("poi", as_index=False)["pre_sum"]
    .sum()
    .rename(columns={"pre_sum": "pre_total"})
)

poi_pre["log_pre"] = np.log1p(poi_pre["pre_total"])

pois = poi_pre["poi"].values
n_pois = len(pois)

poi_to_idx = {p:i for i,p in enumerate(pois)}

In [ ]:
poi_pre.head()

In [ ]:
def build_counterfactual_run(dy_pre, survivors):
    """
    survivors: set of users kept in this run

    Returns:
      dy_cf  : dyad-poi-level counterfactual dataframe
      poi_cf : POI-level totals
    """

    mask = (
        dy_pre["user_i"].isin(survivors) &
        dy_pre["user_j"].isin(survivors)
    )

    dy_cf = dy_pre.loc[mask].copy()
    dy_cf = dy_cf.rename(columns={"pre_sum": "post_cf"})

    # POI-level aggregation
    poi_cf = (
        dy_cf
        .groupby("poi", as_index=False)["post_cf"]
        .sum()
        .rename(columns={"post_cf": "post_total_cf"})
    )

    return dy_cf, poi_cf

In [ ]:
# Universe of users from pre (not strictly needed anymore)
users_all = pd.Index(
    pd.unique(pd.concat([dy_pre["user_i"], dy_pre["user_j"]]))
)

survivor_files = sorted(
    [f for f in os.listdir(survivor_dir) if f.startswith("survivors_run")]
)

for f in survivor_files:

    run = int(f.split("run")[1].split(".")[0])

    with open(os.path.join(survivor_dir, f), "r") as fp:
        survivors = set(json.load(fp))

    # Optional safety: intersect with dy_pre universe
    survivors = survivors.intersection(set(users_all))

    dy_cf, poi_cf = build_counterfactual_run(dy_pre, survivors)

    dy_cf.to_csv(
        os.path.join(cf_out_dir, f"dyad_poi_cf_run{run:04d}.csv"),
        index=False
    )

    poi_cf.to_csv(
        os.path.join(cf_out_dir, f"poi_cf_run{run:04d}.csv"),
        index=False
    )

    print(f"Run {run:04d} saved | dyads={len(dy_cf)} | pois={len(poi_cf)}")

## observed model
 POI-level observed totals + y-hats to be computed only from survivor–survivor interactions, filter dy to dyads where both endpoints are non-evacuated before you aggregate

In [ ]:
dyad_path =  os.path.join(base, "for_ABM", f"individual_interac_allusers_{revision}_aligned.csv")
dy = pd.read_csv(dyad_path)

# Ensure string types
dy["user_i"] = dy["user_i"].astype(str)
dy["user_j"] = dy["user_j"].astype(str)
dy["poi"]    = dy["poi"].astype(str)
dy.columns

In [ ]:
# Ensure evac status numeric (or at least consistent)
dy["evac_status_i"] = pd.to_numeric(dy["evac_status_i"], errors="coerce")
dy["evac_status_j"] = pd.to_numeric(dy["evac_status_j"], errors="coerce")

# ✅ FILTER: keep only survivor-survivor dyads
dy_nonevac = dy.loc[
    (dy["evac_status_i"] == 0) & (dy["evac_status_j"] == 0)
].copy()

print("Rows before:", len(dy))
print("Rows after survivor-only filter:", len(dy_nonevac))

In [ ]:
dy_nonevac.columns

In [ ]:
# # Pre and post sums at dyad-poi level - for full cohort
# dy["pre_sum"]  = dy[["Oct2021", "Nov2021"]].sum(axis=1)
# dy["post_sum"] = dy[["Jan2022", "Feb2022"]].sum(axis=1)

# # Aggregate to POI level
# poi_obs = (
#     dy
#     .groupby("poi", as_index=False)
#     .agg(
#         pre_total=("pre_sum","sum"),
#         post_total=("post_sum","sum"),

#         fire_dist_poi=("fire_dist_poi","first"),
#         edge_poi_dist_mean=("edge_poi_dist_mean","first"),
#         TOP_CATEGORY=("TOP_CATEGORY","first"),
#         SUB_CATEGORY=("SUB_CATEGORY","first"),
#         fire_distance_bin=("fire_distance_bin","first"),
#         polygon_geometry=("polygon_geometry","first"),
#         point_geometry=("point_geometry","first"),

#         # total_population=("total_population","first"),
#         # median_age=("median_age","first"),
#         # white_percent=("white_percent","first"),
#         # median_income_log=("median_income_log","first"),
#         # median_rent=("median_rent","first")
#     )
# )

In [ ]:
# Pre/post sums at dyad-poi level (within survivor universe)
dy_nonevac["pre_sum"]  = dy_nonevac[["Oct2021", "Nov2021"]].sum(axis=1)
dy_nonevac["post_sum"] = dy_nonevac[["Jan2022", "Feb2022"]].sum(axis=1)

# Aggregate to POI level (survivor-only)
poi_obs = (
    dy_nonevac
    .groupby("poi", as_index=False)
    .agg(
        pre_total=("pre_sum","sum"),
        post_total=("post_sum","sum"),

        fire_dist_poi=("fire_dist_poi","first"),
        edge_poi_dist_mean=("edge_poi_dist_mean","first"),
        TOP_CATEGORY=("TOP_CATEGORY","first"),
        SUB_CATEGORY=("SUB_CATEGORY","first"),
        fire_distance_bin=("fire_distance_bin","first"),
        polygon_geometry=("polygon_geometry","first"),
        point_geometry=("point_geometry","first"),
        poi_cbg=("poi_cbg","first")
    )
)


In [ ]:
# Log values
poi_obs["log_pre"]  = np.log1p(poi_obs["pre_total"])
poi_obs["log_post"] = np.log1p(poi_obs["post_total"])
poi_obs["log_delta"] = poi_obs["log_post"] - poi_obs["log_pre"]
poi_obs["delta"] = poi_obs["post_total"] - poi_obs["pre_total"]
poi_obs.head()

# y- hat for observed

In [ ]:
# sdm_cols = [
#     "total_population",
#     "median_age",
#     "white_percent",
#     "median_income_log",
#     "median_rent"
# ]

std_cols = ["fire_dist_poi", "edge_poi_dist_mean"] #+ sdm_cols

# Standardize on observed data
scaler = StandardScaler()
poi_obs[std_cols] = scaler.fit_transform(poi_obs[std_cols])

formula_pre = (
    'pre_total ~ C(SUB_CATEGORY) + '
    + " + ".join(std_cols)
)
formula_pre_log = (
    'log_pre ~ C(SUB_CATEGORY) + '
    + " + ".join(std_cols)
)
formula_post = (
    'post_total ~ C(SUB_CATEGORY) + '
    + " + ".join(std_cols)
)
formula_post_log = (
    'log_post ~ C(SUB_CATEGORY) + '
    + " + ".join(std_cols)
)

formula_delta = (
    'delta ~ C(SUB_CATEGORY) + '
    + " + ".join(std_cols)
)
formula_delta_log = (
    'log_delta ~ C(SUB_CATEGORY) + '
    + " + ".join(std_cols)
)

m_pre   = smf.ols(formula_pre,   data=poi_obs).fit(cov_type="HC3")
m_post = smf.ols(formula_post, data=poi_obs).fit(cov_type="HC3")
m_delta = smf.ols(formula_delta, data=poi_obs).fit(cov_type="HC3")
m_pre_log   = smf.ols(formula_pre_log,   data=poi_obs).fit(cov_type="HC3")
m_post_log = smf.ols(formula_post_log, data=poi_obs).fit(cov_type="HC3")
m_delta_log = smf.ols(formula_delta_log, data=poi_obs).fit(cov_type="HC3")

In [ ]:
poi_obs["yhat_pre"]   = m_pre.predict(poi_obs)
poi_obs["yhat_delta"] = m_delta.predict(poi_obs)
poi_obs["yhat_post"] = m_post.predict(poi_obs)
poi_obs["yhat_log_pre"]   = m_pre_log.predict(poi_obs)
poi_obs["yhat_log_delta"] = m_delta_log.predict(poi_obs)
poi_obs["yhat_log_post"] = m_post_log.predict(poi_obs)
# poi_obs["resid_log_pre"]   = poi_obs["log_pre"]   - poi_obs["yhat_log_pre"]
# poi_obs["resid_log_delta"] = poi_obs["log_delta"] - poi_obs["yhat_log_delta"]
# poi_obs["resid_log_post"] = poi_obs["log_post"] - poi_obs["yhat_log_post"]

In [ ]:
poi_obs.columns

# Counterfactual runs 

In [ ]:
pois = poi_obs["poi"].values
n_pois = len(pois)

poi_to_idx = {p:i for i,p in enumerate(pois)}

log_pre_vec     = poi_obs["log_pre"].values
log_post_vec    = poi_obs["log_post"].values
log_delta_vec   = poi_obs["log_delta"].values
pre_total_vec   = poi_obs["pre_total"].values
post_total_vec  = poi_obs["post_total"].values
delta_obs_vec   = post_total_vec - pre_total_vec
poi_cbg_vec     = poi_obs["poi_cbg"].values

In [ ]:
poi_files = sorted(
    [f for f in os.listdir(cf_out_dir) if f.startswith("poi_cf_run")]
)

B = len(poi_files)
print("Number of runs:", B)

In [ ]:
rows = []

for r, f in enumerate(poi_files):

    poi_cf = pd.read_csv(os.path.join(cf_out_dir, f))
    poi_cf["poi"] = poi_cf["poi"].astype(str)

    # aligned random post vector
    post_cf_vec = np.zeros(n_pois)

    for _, row in poi_cf.iterrows():
        if row["poi"] in poi_to_idx:
            idx = poi_to_idx[row["poi"]]
            post_cf_vec[idx] = row["post_total_cf"]

    # -----------------------------
    # RANDOM quantities
    # -----------------------------
    rand_total = post_cf_vec
    log_rand = np.log1p(rand_total)

    delta_rand = rand_total - pre_total_vec
    log_delta_rand = log_rand - log_pre_vec

    # -----------------------------
    # Build dataframe
    # -----------------------------
    df_run = pd.DataFrame({
        "run": r,
        "poi": pois,
        "poi_cbg": poi_cbg_vec,
        # Observed raw
        "pre_total": pre_total_vec,
        "post_total": post_total_vec,
        "delta_obs": delta_obs_vec,

        # Random raw
        "rand_total": rand_total,
        "delta_rand": delta_rand,

        # Observed logs
        "log_pre": log_pre_vec,
        "log_post": log_post_vec,
        "log_delta": log_delta_vec,

        # Random logs (LEVEL + DELTA)
        "log_rand": log_rand,
        "log_delta_rand": log_delta_rand
    })

    rows.append(df_run)

    if (r+1) % 50 == 0:
        print(f"Processed {r+1}/{B}")

delta_rand_df = pd.concat(rows, ignore_index=True)

print("Final shape:", delta_rand_df.shape)

In [ ]:
delta_rand_df = pd.concat(rows, ignore_index=True)
print(delta_rand_df.columns)
print(delta_rand_df.shape)
delta_rand_df.head()

In [ ]:
output_path = os.path.join(
    cf_out_dir,
    f"poi_pre_with_null_stats_{revision}.csv"
)

delta_rand_df.to_csv(output_path, index=False)

print("Saved:", output_path)

## Merge poi meta with visitation observed and random

In [ ]:
output_path = os.path.join(
    cf_out_dir,
    f"poi_pre_with_null_stats_{revision}.csv"
)

delta_rand_df = pd.read_csv(output_path)

In [ ]:
poi_obs.columns

In [ ]:
delta_rand_df = delta_rand_df.merge(
    poi_obs[
        [
            "poi",
            "edge_poi_dist_mean",
            "TOP_CATEGORY",
            "SUB_CATEGORY",
            "fire_dist_poi",
            "polygon_geometry",
            "point_geometry", 
            'yhat_log_pre',
            'yhat_log_delta',
            'yhat_log_post',
            'yhat_pre',
            'yhat_delta',
            'yhat_post'
        ]
    ],
    on="poi",
    how="left"
)

In [ ]:
delta_rand_df.columns

In [ ]:
output_path = os.path.join(
    cf_out_dir,
    f"poi_with_counterfactual_full_allusers_{revision}.csv"
)

delta_rand_df.to_csv(output_path, index=False)

print("Saved:", output_path)

# Compute y(hats) for random counterfactual

In [ ]:
output_path = os.path.join(
    cf_out_dir,
    f"poi_with_counterfactual_full_allusers_{revision}.csv"
)

delta_rand_df = pd.read_csv(output_path)
print(output_path)

In [ ]:
delta_rand_df.columns

In [ ]:
models_rand_delta = {}
models_rand_post  = {}
models_rand_delta_level = {}
models_rand_post_level  = {}

for r in range(B):

    df_r = delta_rand_df.loc[delta_rand_df["run"] == r].copy()

    # --- model 1: counterfactual log-delta ---
    formula_delta_rand = (
        "log_delta_rand ~ C(SUB_CATEGORY) + " + " + ".join(std_cols)
    )
    m_delta = smf.ols(formula_delta_rand, data=df_r).fit(cov_type="HC3")
    models_rand_delta[r] = m_delta

    delta_rand_df.loc[
        delta_rand_df["run"] == r,
        "yhat_log_delta_rand"
    ] = m_delta.predict(df_r)


    # --- model 2: counterfactual log-post ---
    formula_log_rand = (
        "log_rand ~ C(SUB_CATEGORY) + " + " + ".join(std_cols)
    )
    m_post = smf.ols(formula_log_rand, data=df_r).fit(cov_type="HC3")
    models_rand_post[r] = m_post

    delta_rand_df.loc[
        delta_rand_df["run"] == r,
        "yhat_log_rand"
    ] = m_post.predict(df_r)


    formula_delta_level = (
        "delta_rand ~ C(SUB_CATEGORY) + " + " + ".join(std_cols)
    )
    m_delta_level = smf.ols(formula_delta_level, data=df_r).fit(cov_type="HC3")
    models_rand_delta_level[r] = m_delta_level

    delta_rand_df.loc[
        delta_rand_df["run"] == r,
        "yhat_delta_rand"
    ] = m_delta_level.predict(df_r)


    formula_rand_level = (
        "rand_total ~ C(SUB_CATEGORY) + " + " + ".join(std_cols)
    )
    m_rand_level = smf.ols(formula_rand_level, data=df_r).fit(cov_type="HC3")
    models_rand_post_level[r] = m_rand_level

    delta_rand_df.loc[
        delta_rand_df["run"] == r,
        "yhat_rand"````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````                                                                                                                                                        
    ] = m_rand_level.predict(df_r)

In [ ]:
delta_rand_df.columns

In [ ]:
output_path = os.path.join(
    cf_out_dir,
    f"poi_with_counterfactual_full_allusers_with_yhat_{revision}.csv"
)

delta_rand_df.to_csv(output_path, index=False)

print("Saved:", output_path)

# comuting means for all 500 random versions and saving it in one datafram - random means and st div

In [ ]:
output_path = os.path.join(
    cf_out_dir,
    f"poi_with_counterfactual_full_allusers_with_yhat_{revision}.csv"
)

delta_rand_df = pd.read_csv(output_path)


In [ ]:
delta_rand_df.columns

In [ ]:
rand_summary = (
    delta_rand_df
    .groupby("poi")
    .agg(

        mu_log_delta_rand=("log_delta_rand","mean"),
        sd_log_delta_rand=("log_delta_rand","std"),

        mu_log_rand=("log_rand","mean"),
        sd_log_rand=("log_rand","std"),

        mu_yhat_log_rand=("yhat_log_rand","mean"),
        sd_yhat_log_rand=("yhat_log_rand","std"),

        mu_yhat_log_delta_rand=("yhat_log_delta_rand","mean"),
        sd_yhat_log_delta_rand=("yhat_log_delta_rand","std"),

        mu_delta_rand=("delta_rand","mean"),
        sd_delta_rand=("delta_rand","std"),

        mu_rand=("rand_total","mean"),
        sd_rand=("rand_total","std"),

        mu_yhat_delta_rand=("yhat_delta_rand","mean"),
        sd_yhat_delta_rand=("yhat_delta_rand","std"),

        mu_yhat_rand=("yhat_rand","mean"),
        sd_yhat_rand=("yhat_rand","std"),
    )
    .reset_index()
)

In [ ]:
rand_summary.columns

In [ ]:
poi_obs.columns

In [ ]:
poi_final = poi_obs.merge(
    rand_summary,
    on="poi",
    how="left"
)
poi_final.columns

$z = \frac{\text{Observed} - \text{Random Mean}}{\text{Random SD}}$

In [ ]:
# poi_final["z_rand_log_delta"] = (
#     (poi_final["log_delta"] - poi_final["mu_log_delta_rand"])
#     / poi_final["sd_log_delta_rand"]
# )

In [ ]:
# poi_final["z_rand_delta"] = (
#     (poi_final["delta"] - poi_final["mu_yhat_delta_rand"])
#     / poi_final["sd_yhat_delta_rand"]
# )

In [ ]:
# poi_final["z_yhat_rand_log_delta"] = (
#     (poi_final["log_delta"] - poi_final["mu_yhat_log_delta_rand"])
#     / poi_final["sd_yhat_log_delta_rand"]
# ) 
# #How extreme is the actual structural change relative to the distribution of structural change under random removal


In [ ]:
# z_log = +2 → observed post is 2 SD higher than what your random structural null typically produces (unusually high vs null).
# z_log = −2 → observed post is 2 SD lower than null (unusually low vs null).
# z_log ≈ 0 → observed post is indistinguishable from null expectation.

In [ ]:
poi_final["z"] = (
    (poi_final["post_total"] - poi_final["mu_rand"])
    / poi_final["sd_rand"]
)

In [ ]:
poi_final["z_log"] = (
    (poi_final["log_post"] - poi_final["mu_log_rand"])
    / poi_final["sd_log_rand"]
)

In [ ]:
# poi_final["z_yhat_rand_log"] = (
#     (poi_final["log_post"] - poi_final["mu_yhat_log_rand"])
#     / poi_final["sd_yhat_log_rand"]
# )
# #How extreme is the final post level relative to what random evacuation would produce?


In [ ]:
poi_final.columns

In [ ]:
output_path = os.path.join(
    cf_out_dir,
    f"poi_structural_random_merged_{revision}.csv"
)

poi_final.to_csv(output_path, index=False)

print("Saved:", output_path)

# plotting z

In [ ]:
output_path = os.path.join(
    cf_out_dir,
    f"poi_structural_random_merged_{revision}.csv"
)

poi_final = pd.read_csv(output_path)

In [ ]:
poi_final.columns

In [ ]:
# Keep only POIs with at least 5 pre-disaster visits
poi_final_filtered = poi_final.loc[poi_final["pre_total"] >= 1].copy()

print("Original POIs:", len(poi_final))
print("After filtering (pre_total >= 1):", len(poi_final_filtered))

In [ ]:
# Keep only POIs with at least 5 pre-disaster visits
poi_high_z = poi_final_filtered.loc[poi_final["z"] <= -2].copy()
poi_high_z

In [ ]:
# Keep only POIs with at least 5 pre-disaster visits
poi_high_z = poi_final_filtered.loc[poi_final["z"] >= 15].copy()
poi_high_z

In [ ]:
vals = poi_final_filtered["z_log"].dropna()

plt.figure(figsize=(7,4))

sns.histplot( vals, bins=100, kde=True,color="purple",alpha=0.7)

#plt.yscale("log")
plt.axvline(0, color="black", linestyle="--", linewidth=1)
# plt.title(f"Distribution of prob_shift_joint - {revision}", fontsize=14)
# plt.xlabel("prob_shift_joint ( (post - pre) / pre )", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
vals = poi_final_filtered["z"].dropna()

plt.figure(figsize=(7,4))

sns.histplot( vals, bins=100, kde=True,color="purple",alpha=0.7)

#plt.yscale("log")
plt.axvline(0, color="black", linestyle="--", linewidth=1)
# plt.title(f"Distribution of prob_shift_joint - {revision}", fontsize=14)
# plt.xlabel("prob_shift_joint ( (post - pre) / pre )", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# vals = poi_final["mu_delta_rand"].dropna()
# #true_vals = poi_final["yhat_log_pre"].dropna()
# true_delta = poi_final["delta"].dropna()
# #true_delta = poi_final["yhat_log_delta"].dropna()
# plt.figure(figsize=(7,4))

# sns.histplot( vals, bins=100, kde=True,color="purple",alpha=0.7)
# #sns.histplot( true_vals, bins=100, kde=True,color="green",alpha=0.3)
# sns.histplot( true_delta, bins=100, kde=True,color="orange",alpha=0.5)
# #plt.yscale("log")
# plt.axvline(0, color="black", linestyle="--", linewidth=1)
# # plt.title(f"Distribution of prob_shift_joint - {revision}", fontsize=14)
# # plt.xlabel("prob_shift_joint ( (post - pre) / pre )", fontsize=12)
# plt.ylabel("Count", fontsize=12)
# plt.tight_layout()
# plt.show()

## >0: did better than expected under random removal

## 0: did as expected under the random-removal counterfactual

## <0: worse than expected under random removal

# Plot Code

In [ ]:
from matplotlib.colors import TwoSlopeNorm

In [ ]:
# Convert point geometry
poi_final_filtered["geometry"] = poi_final_filtered["point_geometry"].apply(wkt.loads)

poi_gdf = gpd.GeoDataFrame(
    poi_final_filtered,
    geometry="geometry",
    crs="EPSG:4326"
)

poi_gdf = poi_gdf.set_geometry("geometry")

In [ ]:
geo_dir = os.path.join(base, "geography_CO")
cbg_path = os.path.join(geo_dir, "cbg_colorado_geo.csv")

cbg_df = pd.read_csv(cbg_path)

cbg_df["geometry"] = cbg_df["geometry_wkt"].apply(wkt.loads)

cbg_gdf = gpd.GeoDataFrame(
    cbg_df,
    geometry="geometry",
    crs="EPSG:4326"
)
# Ensure string
cbg_gdf["fips_code"] = cbg_gdf["fips_code"].astype(str)

# Filter Boulder County (08013 → appears as 8013 in your data)
cbg_boulder = cbg_gdf[cbg_gdf["fips_code"].str.startswith("8013")].copy()


In [ ]:
poi_poly_gdf = poi_final.copy()

poi_poly_gdf["geometry"] = poi_poly_gdf["polygon_geometry"].apply(wkt.loads)

poi_poly_gdf = gpd.GeoDataFrame(
    poi_poly_gdf,
    geometry="geometry",
    crs="EPSG:4326"
)

poi_poly_gdf = poi_poly_gdf.to_crs(cbg_boulder.crs)

poi_poly_boulder = gpd.sjoin(
    poi_poly_gdf,
    cbg_boulder[["geometry"]],
    how="inner",
    predicate="intersects"
)

In [ ]:
poi_poly_boulder = gpd.sjoin(
    poi_poly_gdf,
    cbg_boulder[["geometry"]],
    how="inner",
    predicate="intersects"
)

# Also restrict points to Boulder bounding area (cleaner)
poi_points_boulder = gpd.sjoin(
    poi_gdf,
    cbg_boulder[["geometry"]],
    how="inner",
    predicate="intersects"
)

In [ ]:
poi_gdf = poi_gdf.to_crs(cbg_boulder.crs)
poi_poly_gdf = poi_poly_gdf.to_crs(cbg_boulder.crs)

In [ ]:
vmax = abs(poi_points_boulder["z_log"]).max()
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

unique_subcats = poi_points_boulder["SUB_CATEGORY"].unique()

marker_list = [
    "o", "s", "^", "D", "P", "X", "*", "v", "<", ">", "h", "H"
]
marker_map = {
    subcat: marker_list[i % len(marker_list)]
    for i, subcat in enumerate(unique_subcats)
}
# --- Plot ---
fig, ax = plt.subplots(figsize=(12, 12))

#  Background CBGs
cbg_boulder.plot(
    ax=ax,
    color="white",
    edgecolor="lightgrey",
    linewidth=0.4,
    alpha=0.7
)

#  POI polygons (outline only)
poi_poly_boulder.plot(
    ax=ax,
    facecolor="none",     # no fill
    edgecolor="black",
    linewidth=0.7,
    alpha=0.5
)

# POI points (colored by z_log, grouped by SUB_CATEGORY)
for subcat, group in poi_points_boulder.groupby("SUB_CATEGORY"):

    group.plot(
        ax=ax,
        column="z_log",
        cmap="RdYlBu_r",
        norm=norm,
        markersize=10,
        marker=marker_map[subcat],
        alpha=0.5,
        label=subcat
    )

# Colorbar
sm = plt.cm.ScalarMappable(cmap="RdYlBu_r", norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.5)
cbar.set_label("z_log (Post vs Random)")

ax.set_title("Boulder County POIs z_log", fontsize=14)
ax.set_axis_off()

# Optional: legend for shapes
ax.legend(title="SUB_CATEGORY", bbox_to_anchor=(.5, 1), loc="best")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.lines as mlines

# --- Ensure CRS alignment ---
poi_points_boulder = poi_points_boulder.to_crs(cbg_boulder.crs)
poi_poly_boulder = poi_poly_boulder.to_crs(cbg_boulder.crs)

# ----------------------------------------
# 1️⃣ Size scaling (keep on map)
# ----------------------------------------

z_abs = poi_points_boulder["z"].abs()
size_scale = 15
poi_points_boulder["marker_size"] = size_scale * z_abs

# ----------------------------------------
# 2️⃣ Categorical color map
# ----------------------------------------

unique_subcats = poi_points_boulder["SUB_CATEGORY"].unique()
cmap = plt.get_cmap("tab20")

color_map = {
    subcat: cmap(i % 20)
    for i, subcat in enumerate(unique_subcats)
}

poi_points_boulder["color"] = poi_points_boulder["SUB_CATEGORY"].map(color_map)

# ----------------------------------------
# 3️⃣ Plot
# ----------------------------------------

fig, ax = plt.subplots(figsize=(12, 12))

# Background CBGs
cbg_boulder.plot(
    ax=ax,
    color="white",
    edgecolor="lightgrey",
    linewidth=0.4,
    alpha=0.3
)

# POI polygons (outline only)
poi_poly_boulder.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=0.6,
    alpha=0.7
)

# Plot points (size varies on map)
for subcat, group in poi_points_boulder.groupby("SUB_CATEGORY"):

    ax.scatter(
        group.geometry.x,
        group.geometry.y,
        s=group["marker_size"],   # variable size on map
        color=group["color"],
        alpha=0.8,
        marker="o"
    )

# ----------------------------------------
# 4️⃣ Manual Legend (fixed size + counts)
# ----------------------------------------

# Count POIs per category
category_counts = (
    poi_points_boulder
    .groupby("SUB_CATEGORY")
    .size()
    .to_dict()
)

legend_handles = [
    mlines.Line2D(
        [],
        [],
        color=color_map[subcat],
        marker='o',
        linestyle='None',
        markersize=6,   # fixed size in legend
        label=f"{subcat} (n={category_counts[subcat]})"
    )
    for subcat in unique_subcats
]

ax.legend(
    handles=legend_handles,
    title="SUB_CATEGORY",
    bbox_to_anchor=(1.02, 1),
    loc="best",
    frameon=False
)

In [ ]:
import matplotlib.lines as mlines

# --- Ensure CRS alignment ---
poi_points_boulder = poi_points_boulder.to_crs(cbg_boulder.crs)
poi_poly_boulder = poi_poly_boulder.to_crs(cbg_boulder.crs)

# ----------------------------------------
# 1️⃣ Size scaling (keep on map)
# ----------------------------------------

z_abs = poi_points_boulder["z_log"].abs()
size_scale = 20
poi_points_boulder["marker_size"] = size_scale * z_abs

# ----------------------------------------
# 2️⃣ Categorical color map
# ----------------------------------------

unique_subcats = poi_points_boulder["TOP_CATEGORY"].unique()
cmap = plt.get_cmap("tab20")

color_map = {
    subcat: cmap(i % 20)
    for i, subcat in enumerate(unique_subcats)
}

poi_points_boulder["color"] = poi_points_boulder["TOP_CATEGORY"].map(color_map)

# ----------------------------------------
# 3️⃣ Plot
# ----------------------------------------

fig, ax = plt.subplots(figsize=(12, 12))

# Background CBGs
cbg_boulder.plot(
    ax=ax,
    color="white",
    edgecolor="lightgrey",
    linewidth=0.4,
    alpha=0.7
)

# POI polygons (outline only)
poi_poly_boulder.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=0.6,
    alpha=0.4
)

# Plot points (size varies on map)
for subcat, group in poi_points_boulder.groupby("TOP_CATEGORY"):

    ax.scatter(
        group.geometry.x,
        group.geometry.y,
        s=group["marker_size"],   # variable size on map
        color=group["color"],
        alpha=0.4,
        marker="o"
    )

# ----------------------------------------
# 4️⃣ Manual Legend (fixed size + counts)
# ----------------------------------------

# Count POIs per category
category_counts = (
    poi_points_boulder
    .groupby("TOP_CATEGORY")
    .size()
    .to_dict()
)

legend_handles = [
    mlines.Line2D(
        [],
        [],
        color=color_map[subcat],
        marker='o',
        linestyle='None',
        markersize=6,   # fixed size in legend
        label=f"{subcat} (n={category_counts[subcat]})"
    )
    for subcat in unique_subcats
]

ax.legend(
    handles=legend_handles,
    title="TOP_CATEGORY",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

# Regressions

In [ ]:
df = poi_final_filtered.copy()
df.columns

In [ ]:
census_path = os.path.join(geo_dir, "colorado_cbg_census_only.csv")
census = pd.read_csv(census_path)
census["black_percent"] = census["black_alone"] / census["total_population"]
census["white_percent"] = census["white_alone"] / census["total_population"]
census.loc[census["median_income"] < 0, "median_income"] = 1
census.loc[census["median_rent"] < 0, "median_rent"] = 1
census["median_income_log"] = np.log1p(census["median_income"])
census["cbg_fips"] = census["cbg_fips"].astype(str)
census.head()

In [ ]:
df["SUB_CATEGORY"].unique()

In [ ]:
df = df.replace([np.inf, -np.inf], np.nan)
df["poi_cbg"] = df["poi_cbg"].astype(str)
df = df.dropna(subset=["z", "SUB_CATEGORY", "fire_dist_poi", "fire_distance_bin"])
df.shape

In [ ]:
df_c = df.merge(census,
    left_on='poi_cbg',  right_on='cbg_fips',
    how='left')
df_c.head()

In [ ]:
sdm_cols = ["total_population","median_age","white_percent","median_income_log","median_rent"]
std_cols = ["fire_dist_poi", "edge_poi_dist_mean"] + sdm_cols

# Standardize on observed data
scaler = StandardScaler()
df_c[std_cols] = scaler.fit_transform(df_c[std_cols])

In [ ]:
df_c.columns

In [ ]:
# formula = (
#     'mu_rand ~ C(SUB_CATEGORY) + '
#     + " + ".join(std_cols) 
# ) 

# m = smf.ols(formula,   data=df_c).fit()
# print(m.summary())

In [ ]:
# formula = (
#     'log_delta ~ C(SUB_CATEGORY) + '
#     + " + ".join(std_cols) 
# ) 

# m = smf.ols(formula,   data=df_c).fit()
# print(m.summary())

In [ ]:
# formula = (
#     'post_total ~ C(SUB_CATEGORY) + '
#     + " + ".join(std_cols) 
# ) 

# m = smf.ols(formula,   data=df_c).fit()
# print(m.summary())

In [ ]:
# formula = (
#     'pre_total ~ C(SUB_CATEGORY) + '
#     + " + ".join(std_cols) 
# ) 

# m = smf.ols(formula,   data=df_c).fit()
# print(m.summary())

In [ ]:
def extract_model_results(model, model_name):
    """
    Returns a tidy dataframe with:
    Variable | Coefficient | P_value | CI_low | CI_high | R2 | Model
    """
    coef = model.params
    pval = model.pvalues

    # confidence intervals
    ci = model.conf_int()
    ci.columns = ["CI_low", "CI_high"]

    # R2 for OLS, pseudo-R2 for Logit
    if hasattr(model, "rsquared"):
        r2 = float(model.rsquared)
    else:
        r2 = float(model.prsquared)

    # IMPORTANT: align by variable index so CI doesn't become NaN
    df = pd.DataFrame(
        {
            "Variable": coef.index,
            "Coefficient": coef.values,
            "P_value": pval.values,
            "CI_low": ci.loc[coef.index, "CI_low"].values,
            "CI_high": ci.loc[coef.index, "CI_high"].values,
        }
    ).assign(
        Coefficient=lambda d: d["Coefficient"].round(3),
        P_value=lambda d: d["P_value"].round(3),
        CI_low=lambda d: d["CI_low"].round(3),
        CI_high=lambda d: d["CI_high"].round(3),
        R2=round(r2, 3),
        Model=model_name,
    )

    return df

In [ ]:
model_specs = [
    ("OLS_z_log",
     "z_log ~ C(SUB_CATEGORY) + " + " + ".join(std_cols) + " + pre_total"),
    
    ("OLS_log_delta",
     "log_delta ~ C(SUB_CATEGORY) + " + " + ".join(std_cols))
]

all_results = []
fitted_models = {}

for model_name, formula in model_specs:
    print(f"Running model: {model_name}")

    model = smf.ols(formula=formula, data=df_c).fit()

    fitted_models[model_name] = model  # optional: keep models

    res_df = extract_model_results(model, model_name)
    all_results.append(res_df)
results_df = pd.concat(all_results, ignore_index=True)


In [ ]:
results_df

In [ ]:
def plot_coef_with_ci_and_r2_legend(
    results_df,
    *,
    var_order=None,
    var_pretty=None,
    model_pretty=None,
    models_to_plot=None,
    match_on="raw",  # "raw" | "pretty" | "either"
    alpha_sig=0.05,
    grey_non_sig=True,
    nonsig_color="lightgrey",
    nonsig_alpha=0.35,
    sig_alpha=0.95,
    tick_fontsize=12,
    marker_size=70,
    ci_lw=1.6,
    figsize=(12, 7),
    drop_intercept=True,
    drop_category_dummies=True,  # <--- NEW: removes C(SUB_CATEGORY)[T....]
    legend_loc="upper left",
    legend_bbox_to_anchor=(1.02, 1.0),
    legend_fontsize=18,
    pre_color=None,
    delta_color=None,
):
    """
    Single-panel coefficient plot with CIs and R² in legend.
    Expects columns: Variable, Coefficient, CI_low, CI_high, R2, Model
    Optional: significant (bool) else uses P_value < alpha_sig
    """
    df = results_df.copy()

    var_pretty = {} if var_pretty is None else var_pretty
    model_pretty = {} if model_pretty is None else model_pretty

    df["Variable_pretty"] = df["Variable"].map(var_pretty).fillna(df["Variable"])
    df["Model_pretty"] = df["Model"].map(model_pretty).fillna(df["Model"])

    # filter models
    if models_to_plot is not None:
        wanted = set(map(str, models_to_plot))
        if match_on == "raw":
            df = df[df["Model"].astype(str).isin(wanted)].copy()
        elif match_on == "pretty":
            df = df[df["Model_pretty"].astype(str).isin(wanted)].copy()
        elif match_on == "either":
            df = df[
                df["Model"].astype(str).isin(wanted) |
                df["Model_pretty"].astype(str).isin(wanted)
            ].copy()
        else:
            raise ValueError("match_on must be one of {'raw','pretty','either'}")

    if df.shape[0] == 0:
        raise ValueError("After filtering, no rows remain. Check models_to_plot/match_on values.")

    # significance flag
    if "significant" not in df.columns:
        if "P_value" not in df.columns:
            raise ValueError("Need either 'significant' (bool) or 'P_value' (numeric).")
        df["significant"] = pd.to_numeric(df["P_value"], errors="coerce") < float(alpha_sig)

    if drop_intercept:
        df = df[~df["Variable"].isin(["Intercept", "const"])].copy()

    if drop_category_dummies:
        # removes dummy rows like C(SUB_CATEGORY)[T.Some Category]
        df = df[~df["Variable"].astype(str).str.startswith("C(SUB_CATEGORY)[T.")].copy()

    # variable order
    if var_order is None:
        ordered_vars = df["Variable_pretty"].drop_duplicates().tolist()
    else:
        base = [var_pretty.get(v, v) for v in var_order]
        rest = [v for v in df["Variable_pretty"].drop_duplicates().tolist() if v not in base]
        ordered_vars = base + rest

    df["Variable_pretty"] = pd.Categorical(df["Variable_pretty"], categories=ordered_vars, ordered=True)

    # model list
    models = df["Model_pretty"].drop_duplicates().tolist()
    if len(models) == 0:
        raise ValueError("No models to plot after filtering.")

    # two-color scheme (Pre vs Δ)
    if pre_color is None:
        pre_color = "#1f77b4"  # default matplotlib blue
    if delta_color is None:
        delta_color = "#ff7f0e"  # default matplotlib orange

    def is_delta(model_label: str) -> bool:
        s = str(model_label)
        return ("Δ" in s) or s.lower().startswith(("diff", "delta"))

    base_color_map = {m: (delta_color if is_delta(m) else pre_color) for m in models}

    markers = ["o", "s", "D", "^", "v", "P", "X", "*", "<", ">", "h", "H"]
    marker_map = {m: markers[i % len(markers)] for i, m in enumerate(models)}

    # R2 per model
    r2_by_model = df.groupby("Model_pretty")["R2"].max()
    legend_label = {m: f"{m} (R²={r2_by_model.loc[m]:.3f})" for m in models}

    # plot
    fig, ax = plt.subplots(figsize=figsize)

    y_levels = df["Variable_pretty"].cat.categories.tolist()
    y_pos = {v: i for i, v in enumerate(y_levels)}
    df["_y"] = df["Variable_pretty"].map(y_pos).astype(float)

    offsets = np.linspace(-0.20, 0.20, num=max(2, len(models)))
    model_offset = {m: offsets[i] for i, m in enumerate(models)}
    df["_y_dodge"] = df["_y"] + df["Model_pretty"].map(model_offset)

    handles, labels = [], []

    for m in models:
        sub = df[df["Model_pretty"] == m].copy()
        sub_sig = sub[sub["significant"]].copy()
        sub_nsig = sub[~sub["significant"]].copy()

        # CI lines (grey dotted)
        if len(sub_sig):
            ax.hlines(
                y=sub_sig["_y_dodge"],
                xmin=sub_sig["CI_low"],
                xmax=sub_sig["CI_high"],
                linewidth=ci_lw,
                color="grey",
                linestyle=":",
                alpha=0.85,
                zorder=1,
            )
        if len(sub_nsig):
            ax.hlines(
                y=sub_nsig["_y_dodge"],
                xmin=sub_nsig["CI_low"],
                xmax=sub_nsig["CI_high"],
                linewidth=ci_lw,
                color=(nonsig_color if grey_non_sig else "grey"),
                linestyle=":",
                alpha=(nonsig_alpha if grey_non_sig else 0.85),
                zorder=1,
            )

        # points
        if len(sub_nsig):
            ax.scatter(
                sub_nsig["Coefficient"],
                sub_nsig["_y_dodge"],
                s=marker_size,
                marker=marker_map[m],
                color=(nonsig_color if grey_non_sig else base_color_map[m]),
                alpha=(nonsig_alpha if grey_non_sig else sig_alpha),
                edgecolor="none",
                zorder=2,
            )

        if len(sub_sig):
            sc = ax.scatter(
                sub_sig["Coefficient"],
                sub_sig["_y_dodge"],
                s=marker_size,
                marker=marker_map[m],
                color=base_color_map[m],
                alpha=sig_alpha,
                edgecolor="none",
                zorder=3,
            )
        else:
            sc = ax.scatter([], [], s=marker_size, marker=marker_map[m], color=base_color_map[m])

        handles.append(sc)
        labels.append(legend_label[m])

    ax.axvline(0, color="black", linestyle="--", linewidth=1, zorder=0)

    ax.set_yticks(range(len(y_levels)))
    ax.set_yticklabels(y_levels)
    ax.tick_params(axis="x", labelsize=tick_fontsize)
    ax.tick_params(axis="y", labelsize=tick_fontsize)

    ax.set_xlabel("Coefficient", fontsize=tick_fontsize + 1)
    ax.set_ylabel("")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    ax.legend(
        handles,
        labels,
        #title="Model",
        loc=legend_loc,
        bbox_to_anchor=legend_bbox_to_anchor,
        frameon=False,
        fontsize=legend_fontsize,
        title_fontsize=legend_fontsize,
    )

    plt.tight_layout()
    plt.show()

In [ ]:
import re

def pretty_variable_name(v: str) -> str:
    v = str(v)

    if v in ["Intercept", "const"]:
        return "Intercept"

    base_map = {
        "fire_dist_poi": "Distance to fire (m)",
        "edge_poi_dist_mean": "Avg distance from home (m)",
        "median_income_log": "Median income (log)",
        "median_rent": "Median rent",
        "white_percent": "% White",
        "total_population": "Total population",
        "median_age": "Median age",
        "pre_total": "Pre total visits",
    }
    if v in base_map:
        return base_map[v]

    # Match BOTH:
    # 1) C(SUB_CATEGORY)[T.level]
    # 2) C(SUB_CATEGORY, Treatment(reference="X"))[T.level]
    m = re.match(r'^C\((TOP_CATEGORY|SUB_CATEGORY)(?:,\s*Treatment\(reference=.*?\))?\)\[T\.(.*)\]$', v)
    if m:
        #catvar = m.group(1)
        level = m.group(2).strip()

        # optional shortening
        level = level.replace("Restaurants and Other Eating Places", "Restaurants & Cafes")
        level = level.replace("Sporting Goods, Hobby, and Musical Instrument Stores", "Sporting/Hobby/Music")
        level = level.replace("Drinking Places (Alcoholic Beverages)", "Bars & Pubs")
        level = level.replace("Other Amusement and Recreation Industries", "Amusement & Recreation")
        level = level.replace("Museums, Historical Sites, and Similar Institutions", "Museums & Historical")
        level = level.replace("Book Stores and News Dealers", "Bookstores & Newsstands")
        level = level.replace("Other Miscellaneous Store Retailers", "Other retail")
        level = level.replace("Personal Care Services", "Personal Care")

        return f"{level}"

    return v


In [ ]:
results_df["significant"] = results_df["P_value"].astype(float) < 0.05
var_pretty = {v: pretty_variable_name(v) for v in results_df["Variable"].unique()}

model_pretty = {
    "OLS_z_log": "Post: z_log",
    # "OLS_log_delta": "Δ: log_delta",
}

plot_coef_with_ci_and_r2_legend(
    results_df,
    models_to_plot=["OLS_z_log", #"OLS_log_delta"
                   ],
    match_on="raw",
    var_pretty=var_pretty,
    model_pretty=model_pretty,
    tick_fontsize=16,
    figsize=(6, 12),
    drop_category_dummies=False,
)


$z_{\log, i}=\frac{\log(\text{post,i})-\mu(\log(\text{rand,i}))}{\sigma(\log(\text{rand,i}))}$

$z_{\log,i} \sim \beta {c_i} +  D_{f,i} +  D_{\text{poi},i} + \text{SDM}_i $

In your model, each $\beta$ for $ c_i $ is the difference in standardized post-activity (relative to the random expectation) between subcategory and the baseline subcategory

$\Delta_i  \sim \beta {c_i} +  D_{f,i} +  D_{\text{poi},i} + \text{SDM}_i $

$\Delta_i = \log\!\left(\frac{I_{post,i}}{I_{pre,i}}\right)$

$c_i$ = POI subcategory

$D_{f,i}$ = fire distance

$D_{\text{poi},i}$ = network-based POI distance 

$\text{SDM}_i$ = socio-demographic controls 

$\varepsilon_i $ = residual
 
baseline = All Other Amusement and Recreation Industries

Most have higher post visitation compared to expectation, than baseline, except Amusement and Theme Parks which has lower post compare to expectation, than baseline.